# 换一个独立进程，程序还能运行吗

本页学习目标：

- 把 Notebook 定义导出成可以独立启动的 CLI 项目。
- 在新环境中检查新进程实际使用的运行路径。
- 核对成功、失败、退出码和报告授权。
- 明确独立运行检查能够证明什么。

Notebook 内运行通过，可能依赖当前内核残留变量。本页用新进程和新环境启动导出项目，检查交付是否自包含。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. 导出的项目包含什么

```text
project/
  README.md             用途、运行方法和限制
  miniagent/
    __main__.py         命令入口
    _loader.py          导出模块的加载函数
    lesson_*.py         从课程定义生成的 Python 模块
  docs/                 教学资料
  manifest.json         入口、源 Notebook 和代码摘要
```

`collect_definitions` 递归读取带 `definitions` 标签的代码和静态依赖，`export_project` 生成 Python 文件。导出的程序启动时不再读取原 Notebook。

课程 Notebook 仍是维护源代码的位置。需要改功能时先修改定义，再重新导出；手工修改生成文件后，原 manifest 就不能完整解释版本。


## 2. 如何启动独立项目

在下方导出步骤显示的 `project/` 目录打开终端，使用 Python 3.9 或以上版本：

```bash
python3 -m miniagent --question "Agent 如何依据反馈行动？" --output outputs/first-run
```

默认离线模式只需标准库。输出保存到指定目录，读取 `result.json` 检查状态、答案和证据。CLI 不提供网页页面，不需要转发 8888 端口。

若要使用真实模型，启动兼容本课程接口的本地服务，再增加 `--live --model 实际模型名`。本次教学客户端只连接本地回环地址。


## 3. 已有独立验证结果

2026-10-20 的验证在本机新建环境中，移除外部 Python 路径，不继承第三方包，再启动独立进程。

| 用例 | 预期与实际结果 |
| --- | --- |
| 离线正常问题 | 退出码 0，`completed` |
| 无匹配资料 | 退出码 2，`no_evidence` |
| 无效预算参数 | 退出码 2，参数错误 |
| 真实模型正常问题 | 退出码 0，`completed` |
| 极小 token 预算 | 退出码 2，`token_limit` |
| 不存在的模型 | 退出码 2，`model_error` |

六项均符合既定预期。失败用例返回清楚的失败状态，也是验证成功；不是把模型错误算成模型完成任务。

真实模型用例使用本地 CPU 教学服务。新环境仍依赖该服务运行，因此“Python 包自包含”不等于“模型也打包在导出目录中”。


## 4. 授权机制检查了什么

独立子进程还执行六个程序用例：

| 用例 | 要检查的行为 |
| --- | --- |
| 拒绝 | 不写文件 |
| 参数变化 | 不使用旧批准写入 |
| 内容变化 | 不使用旧批准写入 |
| 正确批准 | 创建指定报告 |
| 目标已存在 | 保留原文件，不覆盖 |
| 批准重复使用 | 首次成功，第二次拒绝 |

六项均符合预期。测试给输入函数注入字符串，目的是精确验证程序；真实使用者仍应先查看内容和路径再决定。

这些检查覆盖了列出的分支，没有验证所有文件系统攻击或多用户权限问题。


## 5. 让别人运行时说明什么

独立项目的 README 应直接说明：用户与任务、独立项目中的启动命令、模型服务配置、输出位置、退出码、操作范围和限制。

新增工具时，在课程定义中增加工具说明、实现和权限检查，再导出并回归。不要只在生成目录中修改一份函数，而继续沿用旧源码摘要。

运行失败时保留命令、退出码、状态和 stderr。再次执行是新运行，不能称为从日志恢复。若尚未在另一台机器测试，就明确写本机独立环境验证。


## 6. 练习与代码说明

试着把导出项目复制到新位置，只保留独立项目目录，再用离线命令启动。若要报告跨机器可用，需要在目标机器实际执行并保存结果；本页不替这项工作填写成绩。

默认执行会导出新项目、创建新环境，检查三个离线 CLI 用例和六个授权用例。`RUN_LIVE=False` 时不会重跑真实回答、真实 token 预算或缺失模型三项；正文中的六项 CLI 成绩属于已有历史验证。

生成的新目录位于 `archive/runs/`，使用独立运行标识，不覆盖之前的报告。到这里，课程的工具、检索、状态、评测和交付已经连接起来，可以围绕明确的新任务继续扩展。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
import ast
import hashlib
import importlib
import json
import os
import re
import shutil
import subprocess
import sys
import uuid
import venv
from datetime import datetime
from pathlib import Path

In [ ]:
def collect_definitions(relative, collected=None):
    """从当前仓库的可见 definitions 单元格递归收集静态 Notebook 依赖。"""
    collected = {} if collected is None else collected
    if relative in collected:
        return collected
    path = (ROOT / relative).resolve()
    path.relative_to(ROOT.resolve())
    notebook = json.loads(path.read_text(encoding="utf-8"))
    sources = ["".join(cell["source"]) for cell in notebook["cells"]
               if cell["cell_type"] == "code" and "definitions" in cell.get("metadata", {}).get("tags", [])]
    if not sources:
        raise ValueError("No definitions cells: " + relative)
    source = "\n\n".join(sources)
    collected[relative] = source
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "load_lesson":
            if not node.args or not isinstance(node.args[0], ast.Constant) or not isinstance(node.args[0].value, str):
                raise ValueError("导出只支持静态字符串 load_lesson 依赖：" + relative)
            collect_definitions(node.args[0].value, collected)
    return collected


def export_project(destination):
    """仅复制 definitions 和教学语料；交付包启动时不再读取原 Notebook。"""
    destination = Path(destination).resolve()
    package = destination / "miniagent"
    package.mkdir(parents=True, exist_ok=False)
    entry = "08-capstone/01-research-assistant.ipynb"
    definitions = collect_definitions(entry)
    module_map = {relative: "lesson_" + hashlib.sha256(relative.encode()).hexdigest()[:12]
                  for relative in definitions}
    (package / "__init__.py").write_text('"""Generated from visible teaching Notebook definitions."""\n', encoding="utf-8")
    loader = "\n".join([
        "from pathlib import Path", "import importlib", "",
        "ROOT = Path(__file__).resolve().parents[1]", "OUTPUT = ROOT / 'outputs'",
        "MODULE_MAP = " + repr(module_map), "",
        "def load_lesson(relative):", "    return importlib.import_module('.' + MODULE_MAP[relative], __package__)", "",
    ])
    (package / "_loader.py").write_text(loader, encoding="utf-8")
    for relative, source in definitions.items():
        # __future__ 必须在文件开头；将所有单元格中的 future import 合并到顶部。
        nodes = ast.parse(source).body
        future_nodes = [node for node in nodes if isinstance(node, ast.ImportFrom) and node.module == "__future__"]
        futures = sorted({ast.get_source_segment(source, node) for node in future_nodes})
        lines = source.splitlines()
        skipped = {number for node in future_nodes for number in range(node.lineno, node.end_lineno + 1)}
        body = "\n".join(line for number, line in enumerate(lines, 1) if number not in skipped)
        exported = "\n".join(futures) + "\nfrom ._loader import ROOT, OUTPUT, load_lesson\n\n" + body + "\n"
        compile(exported, module_map[relative], "exec")
        (package / (module_map[relative] + ".py")).write_text(exported, encoding="utf-8")
    (package / "__main__.py").write_text(
        "from ._loader import load_lesson\nraise SystemExit(load_lesson(" + repr(entry) + ").main())\n", encoding="utf-8")
    from miniagent_course import lesson_assets
    source_docs = lesson_assets("02-tools-rag-memory/02-research-agent.ipynb") / "docs"
    shutil.copytree(source_docs, destination / "docs")
    manifest = {"entry": entry, "modules": module_map,
                "source_sha256": {name: hashlib.sha256(source.encode()).hexdigest() for name, source in definitions.items()},
                "source_of_truth": "Notebook definitions cells; generated Python is a delivery artifact"}
    (destination / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    (destination / "README.md").write_text(DELIVERY_README, encoding="utf-8")
    return manifest

In [ ]:
DELIVERY_README = """# Mini Agent · 笔记研究助手

面向阅读学习笔记的同学。输入问题和 docs/ 中的 UTF-8 Markdown，输出带来源的答案、状态和 trace。
本目录由教学 Notebook 的 definitions 单元格导出；可以独立复制，运行不依赖原仓库或 Jupyter。
Python 3.9+，离线与本地 Ollama 模式只使用标准库。不要在生成的 Python 中长期维护课程源码，应修改 Notebook 后重新导出。

## 从空环境运行

在本目录打开终端。创建环境：`python -m venv .venv`。
以下命令中的 python 应使用当前环境的解释器。

```powershell
python -m miniagent --question "Agent 如何依据反馈行动？" --output outputs/run-1
python -m miniagent --question "ZXQVVUNMATCHED" --output outputs/run-2
```

查看对应 outputs/ 中的 result.json。成功退出码 0；无证据、调用失败、预算不足或拒绝导出为 2。
通过进程退出码检查运行结果。无效参数也以 2 退出。
每次运行请使用不同 --output，避免覆盖之前的结果。

## 接入真实模型

安装启动 Ollama，执行 `ollama pull qwen3:1.7b`，再运行：

```powershell
python -m miniagent --live --model qwen3:1.7b --question "Agent 与 workflow 有何区别？" --timeout 120 --output outputs/live
```

服务固定为 127.0.0.1:11434，不需要 API key。离线模式是抽取式检索基线，只有 --live 会调用真实模型。
`--max-steps` 限制模型轮次，`--max-output-tokens` 限制单次生成，总 `--max-tokens` 在请求间检查，是软上限。
暂时网络错误最多重试一次，共用时间预算。本地模型没有 API 美元账单，cost_usd 为 null。
接入远程模型时应新增 adapter，并从环境变量读取凭据；不要把凭据写进 Notebook、报告或 trace。

## 资料、审批与扩展

`--docs` 可以指定语料目录；仅读第一层、非符号链接、64 KB 以下的 Markdown。默认只读教学资料。
加 `--export-report reports/answer.md` 后，终端展示内容、路径及 SHA256，并等待你输入完整批准口令。
拒绝、内容或参数变化、目标已存在都不写出，已经批准过的动作不可重复使用。没有自动发送或发布功能。
要扩展工具，请在课程 Notebook 的 tools 表注册 Tool(schema(...), function)，然后重新导出。
写工具应有具体动作的审批；超时不保证外部动作没有发生，需要另行设计幂等和恢复。

## 限制与核对

检索是词汇基线；引用校验只检查编号，不判断事实是否得到支持。需要结合固定评测任务和人工审阅。
trace 是记录，不是恢复点。本项目是单进程教学作品，没有并发服务、账号权限体系或生产 SLA。
本地新 venv 通过不等于已经验证另一台机器；交接时请记录 Python/模型版本、配置、实际退出码和人工审阅结论。
"""

In [ ]:
run_id = datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]
DELIVERY = OUTPUT / "08-capstone" / ("delivery-" + run_id)
PROJECT = DELIVERY / "project"
manifest = export_project(PROJECT)
print("独立项目：", PROJECT)
print("本次导出的 Notebook：")
for source, module in manifest["modules"].items():
    print(" -", source, "→ miniagent/" + module + ".py")
assert (PROJECT / "docs").is_dir()
assert (PROJECT / "README.md").is_file()

In [ ]:
APPROVAL_CHILD_CODE = r'''
import hashlib
import json
import sys
from pathlib import Path
from miniagent._loader import ROOT, load_lesson

ReportApproval = load_lesson("08-capstone/01-research-assistant.ipynb").ReportApproval
folder = Path(sys.argv[1]).resolve()
folder.mkdir(parents=True)
assert (ROOT / "miniagent").is_dir(), "运行根必须是导出项目"
content = "Only this test report is approved."
token = "APPROVE " + hashlib.sha256(content.encode()).hexdigest()
records = []
for name in ("deny", "wrong_parameters", "changed_content", "approve", "existing", "repeat"):
    target = folder / (name + ".md")
    if name == "existing":
        target.write_text("keep existing", encoding="utf-8")
    action = ReportApproval(target, content, input_fn=lambda _: "no" if name == "deny" else token)
    arguments = {"path": str(target), "sha256": action.digest}
    if name == "wrong_parameters":
        arguments["sha256"] = "changed"
    if name == "changed_content":
        action.content += " changed after preview construction"
    first = action.export(arguments=arguments)
    second = action.export(arguments=arguments) if name == "repeat" else None
    expected = "exported" if name in {"approve", "repeat"} else "denied"
    passed = first["status"] == expected
    if name in {"deny", "wrong_parameters", "changed_content"}:
        passed = passed and not target.exists()
    elif name == "existing":
        passed = passed and target.read_text(encoding="utf-8") == "keep existing"
    else:
        passed = passed and target.read_text(encoding="utf-8") == content
    if name == "repeat":
        passed = passed and second["status"] == "denied"
    records.append({"case": name, "passed": passed, "first": first, "second": second,
                    "input_source": "scripted test response, not actual human approval"})
(folder / "checks.json").write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
raise SystemExit(0 if all(record["passed"] for record in records) else 2)
'''

In [ ]:
def isolated_environment():
    environment = os.environ.copy()
    environment.pop("PYTHONPATH", None)
    environment.pop("PYTHONHOME", None)
    environment["PYTHONNOUSERSITE"] = "1"
    environment["PYTHONIOENCODING"] = "utf-8"
    return environment


def check_delivery(project, run_directory, *, live=False, model="qwen3:1.7b"):
    project, run_directory = Path(project).resolve(), Path(run_directory).resolve()
    environment_directory = run_directory / "venv"
    # 每轮使用新目录，不复用 Notebook kernel 或之前的虚拟环境。
    if environment_directory.exists():
        raise FileExistsError("请使用新的运行目录：" + str(run_directory))
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(environment_directory)
    interpreter = environment_directory / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    environment = isolated_environment()
    identity = subprocess.run([str(interpreter), "-c",
        "import sys,json; print(json.dumps({'executable':sys.executable,'prefix':sys.prefix,'base_prefix':sys.base_prefix,'version':sys.version}))"],
        cwd=project, env=environment, capture_output=True, text=True, encoding="utf-8", timeout=30, check=True)
    interpreter_info = json.loads(identity.stdout)
    assert Path(interpreter_info["executable"]).resolve() == interpreter.resolve()
    assert interpreter_info["prefix"] != interpreter_info["base_prefix"]

    cases = [("offline", ["--question", "Agent 如何依据反馈行动？"], 0, "completed"),
             ("missing", ["--question", "ZXQVVUNMATCHED"], 2, "no_evidence"),
             ("invalid_budget", ["--question", "Agent", "--max-steps", "0"], 2, None)]
    if live:
        cases.extend([
            ("live", ["--live", "--model", model, "--question", "Agent 与 workflow 有何区别？"], 0, "completed"),
            ("live_budget", ["--live", "--model", model, "--max-tokens", "1", "--question", "Agent 如何行动？"], 2, "token_limit"),
            ("missing_model", ["--live", "--model", "miniagent-no-such-model-" + uuid.uuid4().hex,
                               "--question", "Agent 如何行动？"], 2, "model_error"),
        ])
    records = []
    for name, options, expected_exit, expected_status in cases:
        target = run_directory / "cases" / name
        command = [str(interpreter), "-m", "miniagent", *options, "--output", str(target)]
        completed = subprocess.run(command, cwd=project, env=environment, capture_output=True,
                                   text=True, encoding="utf-8", timeout=150)
        path = target / "result.json"
        payload = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
        passed = completed.returncode == expected_exit and (expected_status is None or payload.get("status") == expected_status)
        records.append({"case": name, "command": command, "exit_code": completed.returncode,
                        "status": payload.get("status"), "passed": passed,
                        "stdout": completed.stdout, "stderr": completed.stderr})
        print(name, "PASS" if passed else "FAIL", "exit=", completed.returncode, "status=", payload.get("status"))

    # 审批同样交给独立解释器。脚本内容保留在 Notebook 中，导出目录只接收运行产物。
    approval_code = APPROVAL_CHILD_CODE
    approval_result = subprocess.run([str(interpreter), "-c", approval_code, str(run_directory / "approval")],
        cwd=project, env=environment, capture_output=True, text=True, encoding="utf-8", timeout=30)
    approval_path = run_directory / "approval" / "checks.json"
    approval_checks = json.loads(approval_path.read_text(encoding="utf-8")) if approval_path.exists() else []
    outcome = {"interpreter": interpreter_info, "project": str(project),
               "isolation": "fresh venv, no third-party site packages, exported project only; subprocess execution",
               "live_requested": live, "cli_runs": records, "approval_cases": approval_checks,
               "approval_exit_code": approval_result.returncode,
               "approval_stdout": approval_result.stdout, "approval_stderr": approval_result.stderr}
    outcome["passed"] = (all(record["passed"] for record in records) and approval_result.returncode == 0
                         and len(approval_checks) == 6 and all(record["passed"] for record in approval_checks))
    (run_directory / "delivery-result.json").write_text(json.dumps(outcome, ensure_ascii=False, indent=2), encoding="utf-8")
    assert outcome["passed"], "交付检查失败；先查看 delivery-result.json 中真实 stderr/status，再修复。"
    return outcome

In [ ]:
delivery = check_delivery(PROJECT, DELIVERY)
print("交付验证：", delivery["passed"])
print("实际解释器：", delivery["interpreter"]["executable"])
print("CLI 用例：", len(delivery["cli_runs"]), "；审批用例：", len(delivery["approval_cases"]))
print("完整证据：", DELIVERY / "delivery-result.json")

In [ ]:
RUN_LIVE = False
MODEL = "qwen3:1.7b"
if RUN_LIVE:
    live_id = datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]
    LIVE_DELIVERY = OUTPUT / "08-capstone" / ("live-delivery-" + live_id)
    export_project(LIVE_DELIVERY / "project")
    live_delivery = check_delivery(LIVE_DELIVERY / "project", LIVE_DELIVERY, live=True, model=MODEL)
    print("真实交付证据：", LIVE_DELIVERY / "delivery-result.json")
else:
    print("本次只验证离线交付；未声称真实模型验收通过。")